In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [2]:

df = pd.read_csv("../data/truck_market_dataset_10000.csv")

print(df.shape)
print(df.head())
print(df.info())

(10000, 22)
   year       make            model      trim         body transmission_type  \
0  2014  Chevrolet   Impala Limited  LT Fleet        Sedan         automatic   
1  2003      Dodge  Ram Pickup 1500       SLT     Quad Cab               NaN   
2  2007    Pontiac               G6        GT  Convertible         automatic   
3  2011     Toyota          Corolla        LE        Sedan         automatic   
4  2012      Lexus           ES 350      Base        Sedan               NaN   

                 vin location  condition  mileage  ...  \
0  2g1wb5e37e1112559       fl        4.0  21507.0  ...   
1  1d7ha18n13s152972       mo       31.0  79712.0  ...   
2  1g2zh361474252178       nj       34.0  65698.0  ...   
3  jtdbu4eexb9167571       fl       43.0  23634.0  ...   
4  jthbk1eg6c2495519       pa       35.0  26483.0  ...   

                             seller      mmr  price  \
0                    gm remarketing  13450.0  13800   
1                  tdaf remarketing   6025.0   6

In [3]:
df = df[df["price"] > 1000]
df.shape

(9691, 22)

In [4]:
# average price for each make-model pair
model_avg = df.groupby(
    ["make", "model"]
)["price"].mean().reset_index()

model_avg.columns = [
    "make",
    "model",
    "avg_model_price"
]

# merge back
df = df.merge(
    model_avg,
    on=["make", "model"],
    how="left"
)

# relative brand factor
df["brand_factor"] = (
    df["price"] / df["avg_model_price"]
)

In [5]:
df = df[df["mileage"] > 100]
df.shape

(9664, 24)

In [6]:
core_cols = [
    "make",
    "model",
    "year",
    "price",
    "mileage"
]

df = df.dropna(subset=core_cols)

In [7]:
df.head()

,year,make,model,trim,body,transmission_type,vin,location,condition,mileage,...,price,auction_date,source,source_type,truck_type,first_seen_date,status,last_seen_date,avg_model_price,brand_factor
0,2014,Chevrolet,Impala Limited,LT Fleet,Sedan,automatic,2g1wb5e37e1112559,fl,4.0,21507.0,...,13800,2015-02-22 21:00:00+00:00,CommercialTruckTrader,retail,Day Cab,2014-11-11 21:00:00+00:00,ended,2015-02-22 21:00:00+00:00,13778.947368,1.001528
1,2003,Dodge,Ram Pickup 1500,SLT,Quad Cab,NaN,1d7ha18n13s152972,mo,31.0,79712.0,...,6300,2015-01-19 18:30:00+00:00,IronPlanet,retail,Sleeper,2014-12-05 18:30:00+00:00,ended,2015-01-19 18:30:00+00:00,9377.142857,0.671846
2,2007,Pontiac,G6,GT,Convertible,automatic,1g2zh361474252178,nj,34.0,65698.0,...,8000,2015-01-13 17:30:00+00:00,IronPlanet,retail,Flatbed,2014-12-03 17:30:00+00:00,ended,2015-01-13 17:30:00+00:00,4266.666667,1.875000
3,2011,Toyota,Corolla,LE,Sedan,automatic,jtdbu4eexb9167571,fl,43.0,23634.0,...,11400,2015-01-26 17:30:00+00:00,TruckPaper,retail,Reefer,2014-12-01 17:30:00+00:00,active,2015-01-26 17:30:00+00:00,9812.780142,1.161750
4,2012,Lexus,ES 350,Base,Sedan,NaN,jthbk1eg6c2495519,pa,35.0,26483.0,...,23300,2015-01-29 17:00:00+00:00,CommercialTruckTrader,retail,Reefer,2014-12-28 17:00:00+00:00,ended,2015-01-29 17:00:00+00:00,20322.857143,1.146492


In [8]:
CURRENT_YEAR = 2026

df["age"] = CURRENT_YEAR - df["year"]

In [9]:
df["age"].describe()

count    9506.000000
mean       15.561750
std         3.562861
min        11.000000
25%        13.000000
50%        14.000000
75%        18.000000
max        35.000000
Name: age, dtype: float64

In [10]:
df["price_per_mile"] = (
    df["price"] /
    (df["mileage"]+1)
)

In [11]:
df["first_seen_date"] = pd.to_datetime(
    df["first_seen_date"]
)

df["last_seen_date"] = pd.to_datetime(
    df["last_seen_date"]
)

df["days_on_market"] = (
    df["last_seen_date"] -
    df["first_seen_date"]
).dt.days

In [12]:
df["days_on_market"].describe()

count    9506.000000
mean       63.655481
std        32.456315
min         7.000000
25%        36.000000
50%        64.000000
75%        92.000000
max       119.000000
Name: days_on_market, dtype: float64

In [13]:
base_features = [
    'body',
    'transmission_type',
    'seller',
    'source_type',
    'truck_type',
    'status',
    'age',
    'mileage',
    'days_on_market',
]

In [14]:
df = df.dropna(subset=core_cols)

In [15]:


X_base = df[base_features]

y_base = df["price"]

In [16]:
cat_base = [
    'body',
    'transmission_type',
    'seller',
    'source_type',
    'truck_type',
    'status',
]

In [17]:
num_base=[
    'age',
    'mileage',
    'days_on_market',

]

In [24]:
df[base_features].isnull().sum()

body                 0
transmission_type    0
seller               0
source_type          0
truck_type           0
status               0
age                  0
mileage              0
days_on_market       0
dtype: int64

In [19]:
df["body"] = df["body"].fillna("Unknown")
df["transmission_type"] = df["transmission_type"].fillna("Unknown")

In [20]:
df['days_on_market'] = df['days_on_market'].fillna(df['days_on_market'].median())

In [21]:
missing_pct = df[num_base+cat_base].isnull().mean() * 100
print(missing_pct)

age                  0.0
mileage              0.0
days_on_market       0.0
body                 0.0
transmission_type    0.0
seller               0.0
source_type          0.0
truck_type           0.0
status               0.0
dtype: float64


In [25]:
X_base = df[base_features]

y_base = df["price"]

In [26]:
from catboost import CatBoostRegressor


base_model = CatBoostRegressor(
    iterations=1000,
    depth=4,
    learning_rate=0.03,
    loss_function="RMSE",
    verbose=100
)


base_model.fit(
    X_base,
    y_base,
    cat_features=cat_base
)

0:	learn: 9416.7182355	total: 48.8ms	remaining: 48.7s
100:	learn: 6447.8056275	total: 202ms	remaining: 1.8s
200:	learn: 6236.5240754	total: 353ms	remaining: 1.4s
300:	learn: 6169.3894063	total: 498ms	remaining: 1.16s
400:	learn: 6129.1175356	total: 646ms	remaining: 965ms
500:	learn: 6086.8682554	total: 785ms	remaining: 782ms
600:	learn: 6037.7123259	total: 950ms	remaining: 630ms
700:	learn: 6004.1112676	total: 1.1s	remaining: 472ms
800:	learn: 5965.9578327	total: 1.25s	remaining: 311ms
900:	learn: 5929.6626548	total: 1.4s	remaining: 154ms
999:	learn: 5896.4311125	total: 1.57s	remaining: 0us


CatBoostRegressor(depth=4, iterations=1000, learning_rate=0.03, loss_function='RMSE', verbose=100)

In [27]:
brand_features = [
    "make",
    "model"
]

X_brand = df[brand_features]

y_brand = df["brand_factor"]

In [30]:
cat_brand = [
    "make",
    "model"
]

In [31]:
brand_model = CatBoostRegressor(
    iterations=500,
    depth=3,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100
)


brand_model.fit(
    X_brand,
    y_brand,
    cat_features=cat_brand
)

0:	learn: 0.4411390	total: 1.51ms	remaining: 756ms
100:	learn: 0.4385645	total: 78ms	remaining: 308ms
200:	learn: 0.4378692	total: 150ms	remaining: 223ms
300:	learn: 0.4374092	total: 227ms	remaining: 150ms
400:	learn: 0.4369477	total: 293ms	remaining: 72.4ms
499:	learn: 0.4365461	total: 362ms	remaining: 0us


CatBoostRegressor(depth=3, iterations=500, learning_rate=0.05, loss_function='RMSE', verbose=100)

In [36]:
def predict_vehicle_price(
    make,
    model,
    age,
    mileage,
    body="Sedan",
    transmission_type="Automatic",
    seller="Dealer",
    source_type="retail",
    truck_type="Standard",
    status="active",
    days_on_market=30
):

    base_input = pd.DataFrame([{
        "body": body,
        "transmission_type": transmission_type,
        "seller": seller,
        "source_type": source_type,
        "truck_type": truck_type,
        "status": status,
        "age": age,
        "mileage": mileage,
        "days_on_market": days_on_market
    }])

    base_price = base_model.predict(base_input)[0]

    brand_input = pd.DataFrame([{
        "make": make,
        "model": model
    }])

    factor = brand_model.predict(brand_input)[0]

    final_price = base_price * factor

    return final_price

In [35]:
print(base_model.get_cat_feature_indices())

[0, 1, 2, 3, 4, 5]


In [37]:
for age in [2,4,6,8,10]:

    pred = predict_vehicle_price(
        make="Honda",
        model="Civic",
        age=age,
        mileage=20000,
        body="Sedan",
        transmission_type="Automatic"
    )

    print(age, pred)

2 18919.684129804282
4 18919.684129804282
6 18919.684129804282
8 18919.684129804282
10 18919.684129804282


In [38]:
print(base_model.feature_importances_)
print(base_model.feature_names_)

[27.62937644  1.04242608 20.08586806  0.05113023  0.92455818  0.73514763
 15.71433313 33.36460231  0.45255796]
['body', 'transmission_type', 'seller', 'source_type', 'truck_type', 'status', 'age', 'mileage', 'days_on_market']
